In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = "/usr/lib/jvm/java-11-openjdk-amd64/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("nyc_taxi_explore") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(spark.version)

your 131072x1 screen size is bogus. expect trouble
26/07/25 17:25:25 WARN Utils: Your hostname, nhuaaaaaaa resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/25 17:25:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/25 17:25:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.3


In [2]:
# ============================================================
# Cell 2 — Load 2 file đại diện và so sánh schema
# ============================================================

df_2011 = spark.read.parquet("../data/raw/yellow_tripdata_2011-01.parquet")
df_2024 = spark.read.parquet("../data/raw/yellow_tripdata_2024-01.parquet")

print("=== 2011 schema ===")
df_2011.printSchema()

print(f"\n2011 row count: {df_2011.count():,}")

print("\n=== 2024 schema ===")
df_2024.printSchema()

print(f"\n2024 row count: {df_2024.count():,}")

=== 2011 schema ===
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)


2011 row count: 13,464,997

=== 2024 schema ===
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_da

In [3]:

print("\n=== 2024 schema ===")
df_2024.printSchema()


=== 2024 schema ===
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [4]:
# ============================================================
# Cell 3 — Data quality statistics (dùng để định ngưỡng validate Phase 3)
# ============================================================
from pyspark.sql import functions as F

def quality_stats(df, year_label):
    total = df.count()
    
    stats = df.agg(
        # Fare issues
        F.count(F.when(F.col("fare_amount") < 0, 1)).alias("fare_negative"),
        F.count(F.when(F.col("fare_amount") == 0, 1)).alias("fare_zero"),
        
        # Distance issues
        F.count(F.when(F.col("trip_distance") <= 0, 1)).alias("distance_lte_zero"),
        
        # Null location
        F.count(F.when(F.col("PULocationID").isNull(), 1)).alias("null_pickup_location"),
        F.count(F.when(F.col("DOLocationID").isNull(), 1)).alias("null_dropoff_location"),
        
        # Duration issues (tính bằng phút)
        F.count(F.when(
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60 <= 0, 1
        )).alias("duration_lte_zero"),
        F.count(F.when(
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60 > 1440, 1
        )).alias("duration_gt_24h"),
        
        # Fare stats
        F.min("fare_amount").alias("fare_min"),
        F.max("fare_amount").alias("fare_max"),
        F.percentile_approx("fare_amount", [0.01, 0.25, 0.5, 0.75, 0.99]).alias("fare_percentiles"),
        
        # Distance stats
        F.min("trip_distance").alias("dist_min"),
        F.max("trip_distance").alias("dist_max"),
        F.percentile_approx("trip_distance", [0.01, 0.25, 0.5, 0.75, 0.99]).alias("dist_percentiles"),
    ).collect()[0]
    
    print(f"\n{'='*50}")
    print(f"  {year_label}  |  Total rows: {total:,}")
    print(f"{'='*50}")
    print(f"[FARE]")
    print(f"  negative    : {stats['fare_negative']:,} ({stats['fare_negative']/total*100:.2f}%)")
    print(f"  zero        : {stats['fare_zero']:,} ({stats['fare_zero']/total*100:.2f}%)")
    print(f"  min/max     : {stats['fare_min']} / {stats['fare_max']}")
    print(f"  percentiles : p1={stats['fare_percentiles'][0]}, p25={stats['fare_percentiles'][1]}, "
          f"p50={stats['fare_percentiles'][2]}, p75={stats['fare_percentiles'][3]}, p99={stats['fare_percentiles'][4]}")
    print(f"[DISTANCE]")
    print(f"  lte zero    : {stats['distance_lte_zero']:,} ({stats['distance_lte_zero']/total*100:.2f}%)")
    print(f"  min/max     : {stats['dist_min']} / {stats['dist_max']}")
    print(f"  percentiles : p1={stats['dist_percentiles'][0]}, p25={stats['dist_percentiles'][1]}, "
          f"p50={stats['dist_percentiles'][2]}, p75={stats['dist_percentiles'][3]}, p99={stats['dist_percentiles'][4]}")
    print(f"[LOCATION]")
    print(f"  null pickup : {stats['null_pickup_location']:,} ({stats['null_pickup_location']/total*100:.2f}%)")
    print(f"  null dropoff: {stats['null_dropoff_location']:,} ({stats['null_dropoff_location']/total*100:.2f}%)")
    print(f"[DURATION]")
    print(f"  lte zero    : {stats['duration_lte_zero']:,} ({stats['duration_lte_zero']/total*100:.2f}%)")
    print(f"  gt 24h      : {stats['duration_gt_24h']:,} ({stats['duration_gt_24h']/total*100:.2f}%)")

quality_stats(df_2011, "yellow_2011-01")
quality_stats(df_2024, "yellow_2024-01")


  yellow_2011-01  |  Total rows: 13,464,997
[FARE]
  negative    : 0 (0.00%)
  zero        : 0 (0.00%)
  min/max     : 2.5 / 500.0
  percentiles : p1=3.3, p25=5.7, p50=7.7, p75=10.9, p99=45.0
[DISTANCE]
  lte zero    : 76,093 (0.57%)
  min/max     : 0.0 / 100.0
  percentiles : p1=0.1, p25=1.0, p50=1.69, p75=3.0, p99=17.6
[LOCATION]
  null pickup : 0 (0.00%)
  null dropoff: 0 (0.00%)
[DURATION]
  lte zero    : 23,148 (0.17%)
  gt 24h      : 4 (0.00%)



  yellow_2024-01  |  Total rows: 2,964,624
[FARE]
  negative    : 37,448 (1.26%)
  zero        : 893 (0.03%)
  min/max     : -899.0 / 5000.0
  percentiles : p1=-5.1, p25=8.6, p50=12.8, p75=20.5, p99=76.2
[DISTANCE]
  lte zero    : 60,371 (2.04%)
  min/max     : 0.0 / 312722.3
  percentiles : p1=0.0, p25=1.0, p50=1.68, p75=3.11, p99=20.0
[LOCATION]
  null pickup : 0 (0.00%)
  null dropoff: 0 (0.00%)
[DURATION]
  lte zero    : 870 (0.03%)
  gt 24h      : 16 (0.00%)


In [6]:
# ============================================================
# Cell 4 — Check LocationID validity vs zone lookup
# ============================================================
import pandas as pd

# Đọc zone lookup
zone_pdf = pd.read_csv("../data/raw/taxi_zone_lookup.csv")
print("Zone lookup shape:", zone_pdf.shape)
print(zone_pdf.head())
print("\nLocationID range:", zone_pdf["LocationID"].min(), "→", zone_pdf["LocationID"].max())
print("Unique boroughs:", zone_pdf["Borough"].unique())

# Convert sang Spark DataFrame để join check
zone_df = spark.createDataFrame(zone_pdf)
valid_ids = set(zone_pdf["LocationID"].tolist())
print(f"\nTotal valid LocationIDs: {len(valid_ids)}")

# Check invalid LocationID trong trip data
for label, df in [("2011", df_2011), ("2024", df_2024)]:
    total = df.count()
    invalid_pu = df.filter(~F.col("PULocationID").isin(valid_ids)).count()
    invalid_do = df.filter(~F.col("DOLocationID").isin(valid_ids)).count()
    print(f"\n[{label}]")
    print(f"  Invalid PULocationID: {invalid_pu:,} ({invalid_pu/total*100:.2f}%)")
    print(f"  Invalid DOLocationID: {invalid_do:,} ({invalid_do/total*100:.2f}%)")

Zone lookup shape: (265, 4)
   LocationID        Borough                     Zone service_zone
0           1            EWR           Newark Airport          EWR
1           2         Queens              Jamaica Bay    Boro Zone
2           3          Bronx  Allerton/Pelham Gardens    Boro Zone
3           4      Manhattan            Alphabet City  Yellow Zone
4           5  Staten Island            Arden Heights    Boro Zone

LocationID range: 1 → 265
Unique boroughs: ['EWR' 'Queens' 'Bronx' 'Manhattan' 'Staten Island' 'Brooklyn' 'Unknown'
 nan]

Total valid LocationIDs: 265



[2011]
  Invalid PULocationID: 0 (0.00%)
  Invalid DOLocationID: 0 (0.00%)



[2024]
  Invalid PULocationID: 0 (0.00%)
  Invalid DOLocationID: 0 (0.00%)


In [7]:
# ============================================================
# Cell 5 — Duration percentiles để chọn ngưỡng hợp lý
# ============================================================
for label, df in [("2011", df_2011), ("2024", df_2024)]:
    duration_stats = df.select(
        F.percentile_approx(
            (F.unix_timestamp("tpep_dropoff_datetime") - 
             F.unix_timestamp("tpep_pickup_datetime")) / 60,
            [0.01, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
        ).alias("duration_percentiles")
    ).collect()[0]["duration_percentiles"]
    
    print(f"[{label}] duration (minutes):")
    print(f"  p1={duration_stats[0]:.1f}, p25={duration_stats[1]:.1f}, "
          f"p50={duration_stats[2]:.1f}, p75={duration_stats[3]:.1f}, "
          f"p95={duration_stats[4]:.1f}, p99={duration_stats[5]:.1f}, "
          f"p99.9={duration_stats[6]:.1f}")

[2011] duration (minutes):
  p1=1.4, p25=6.0, p50=9.8, p75=15.0, p95=27.3, p99=42.6, p99.9=69.8
[2024] duration (minutes):
  p1=0.6, p25=7.2, p50=11.6, p75=18.7, p95=37.9, p99=60.4, p99.9=115.2
